# Two-Tower Business Entity Resolution: Kaggle GPU Training & Inference

### Workflow Overview:
1. Attach your preprocessed dataset in Kaggle (`/kaggle/input/...`).
2. Turn on GPU accelerator: **Settings -> Accelerator -> GPU T4 x2 or P100**.
3. Run training with PyTorch Mixed Precision (AMP) for ultra-fast GPU throughput.
4. Trained weights (`best.pt`, `tokenizer.json`, `legal_form_vocab.json`, etc.) are saved to `/kaggle/working/two_tower_model`.
5. An automated zip archive (`two_tower_model.zip`) is created for easy download to your local machine.
6. Verify and run predictions directly in the next cells using the saved weights.


In [4]:
import os
import sys
import time
import json
import re
import shutil
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

# Compute device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"[INFO] Using device: {device}")
if device.type == 'cuda':
    print(f"[INFO] GPU Name: {torch.cuda.get_device_name(0)}")

# Helper to automatically locate files in /kaggle/input if dataset name varies
def locate_file(pattern, search_root=Path('/kaggle/input')):
    matches = list(search_root.rglob(pattern)) if search_root.exists() else []
    return matches[0] if matches else None

# Dataset paths: auto-detected from /kaggle/input or set manually
SOURCE1_PATH = locate_file('*sounce1*.tsv') or Path(r'C:\Users\ps302\OneDrive\Desktop\Amazon\business-entity-resolution\data\pre-processing\preprocessed_train_sounce1.tsv')
SOURCE2_PATH = locate_file('*sounce2*.tsv') or Path(r'C:\Users\ps302\OneDrive\Desktop\Amazon\business-entity-resolution\data\pre-processing\preprocessed_train_sounce2.tsv')
SOURCE3_PATH = locate_file('*sounce3*.tsv') or Path(r'C:\Users\ps302\OneDrive\Desktop\Amazon\business-entity-resolution\data\pre-processing\preprocessed_train_sounce3.tsv')
LABELS_PATH  = locate_file('*labeledPairs*.tsv') or Path(r'C:\Users\ps302\OneDrive\Desktop\Amazon\business-entity-resolution\data\pre-processing\labeledPairs.tsv')

OUTPUT_DIR = Path(r'C:\Users\ps302\OneDrive\Desktop\Amazon\business-entity-resolution\data\model_output')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Training configurations for Kaggle GPU
EPOCHS = 5
BATCH_SIZE = 256  # 256 or 512 for GPU
LEARNING_RATE = 1e-3
MAX_LABEL_ROWS = None  # Use None for full dataset, or e.g. 500000 for quick test

print(f"[INFO] Source 1 Path: {SOURCE1_PATH}")
print(f"[INFO] Source 2 Path: {SOURCE2_PATH}")
print(f"[INFO] Source 3 Path: {SOURCE3_PATH}")
print(f"[INFO] Labels Path  : {LABELS_PATH}")
print(f"[INFO] Output Dir   : {OUTPUT_DIR}")


[INFO] Using device: cpu
[INFO] Source 1 Path: C:\Users\ps302\OneDrive\Desktop\Amazon\business-entity-resolution\data\pre-processing\preprocessed_train_sounce1.tsv
[INFO] Source 2 Path: C:\Users\ps302\OneDrive\Desktop\Amazon\business-entity-resolution\data\pre-processing\preprocessed_train_sounce2.tsv
[INFO] Source 3 Path: C:\Users\ps302\OneDrive\Desktop\Amazon\business-entity-resolution\data\pre-processing\preprocessed_train_sounce3.tsv
[INFO] Labels Path  : C:\Users\ps302\OneDrive\Desktop\Amazon\business-entity-resolution\data\pre-processing\labeledPairs.tsv
[INFO] Output Dir   : C:\Users\ps302\OneDrive\Desktop\Amazon\business-entity-resolution\data\model_output


In [2]:
# ==============================================================================
# Model Architecture & Utilities (Self-Contained)
# ==============================================================================

TEXT_FIELDS = ("name_norm", "name_ascii", "address_norm", "country_norm")

def split_legal_forms(value: str) -> list[str]:
    value = str(value or "").strip()
    if value.startswith("["):
        try:
            value = ",".join(json.loads(value))
        except (json.JSONDecodeError, TypeError):
            pass
    strip_chars = " '\""
    tokens = re.split(r"[,;|]", value)
    result = []
    for x in tokens:
        cleaned = x.strip(strip_chars)
        if cleaned and cleaned != "missing":
            result.append(cleaned)
    return result



class CharTokenizer:
    PAD, UNK = 0, 1

    def __init__(self, vocab: dict[str, int]):
        self.vocab = vocab

    @classmethod
    def fit(cls, records: pd.DataFrame, max_chars: int = 12000) -> "CharTokenizer":
        counts = Counter()
        for field in TEXT_FIELDS:
            counts.update("".join(records[field].fillna("").astype(str)))
        vocab = {char: index + 2 for index, (char, _) in
                 enumerate(counts.most_common(max_chars))}
        return cls(vocab)

    def encode(self, values, length: int) -> torch.Tensor:
        output = np.zeros((len(values), length), dtype=np.int64)
        for row, value in enumerate(values):
            ids = [self.vocab.get(char, self.UNK) for char in str(value)[:length]]
            output[row, :len(ids)] = ids
        return torch.from_numpy(output)

    def save(self, path):
        Path(path).write_text(json.dumps(self.vocab, ensure_ascii=False), encoding="utf-8")

    @classmethod
    def load(cls, path):
        return cls(json.loads(Path(path).read_text(encoding="utf-8")))


class LegalFormVocab:
    PAD, UNK = 0, 1

    def __init__(self, vocab: dict[str, int]):
        self.vocab = vocab

    @classmethod
    def fit(cls, records: pd.DataFrame) -> "LegalFormVocab":
        values = sorted({form for value in records["legal_form"].fillna("")
                         for form in split_legal_forms(value)})
        return cls({value: index + 2 for index, value in enumerate(values)})

    def encode(self, values) -> torch.Tensor:
        forms = [split_legal_forms(value) for value in values]
        width = max(1, max(map(len, forms), default=0))
        output = np.zeros((len(forms), width), dtype=np.int64)
        for row, values_for_row in enumerate(forms):
            if values_for_row:
                output[row, :len(values_for_row)] = [self.vocab.get(value, self.UNK) for value in values_for_row]
        return torch.from_numpy(output)

    def save(self, path):
        Path(path).write_text(json.dumps(self.vocab, indent=2), encoding="utf-8")

    @classmethod
    def load(cls, path):
        return cls(json.loads(Path(path).read_text(encoding="utf-8")))


class CharCNN(nn.Module):
    def __init__(self, vocab_size: int, char_dim: int = 48, channels: int = 64, output_dim: int = 128):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, char_dim, padding_idx=0)
        self.convs = nn.ModuleList([
            nn.Conv1d(char_dim, channels, kernel_size=kernel, padding=kernel // 2)
            for kernel in (3, 5, 7)
        ])
        self.projection = nn.Linear(channels * 3, output_dim)

    def forward(self, token_ids: torch.Tensor) -> torch.Tensor:
        present = token_ids.ne(0)
        x = self.embedding(token_ids).transpose(1, 2)
        pooled = []
        for conv in self.convs:
            value = F.relu(conv(x)).masked_fill(~present.unsqueeze(1), -1e4)
            value = value.max(dim=-1).values
            pooled.append(torch.where(present.any(1, keepdim=True), value, torch.zeros_like(value)))
        return self.projection(torch.cat(pooled, dim=1))


class EntityEncoder(nn.Module):
    def __init__(self, char_vocab_size: int, legal_vocab_size: int,
                 char_dim: int = 48, channels: int = 64, field_dim: int = 128,
                 legal_dim: int = 16, output_dim: int = 128, dropout: float = 0.15):
        super().__init__()
        self.name_cnn = CharCNN(char_vocab_size, char_dim, channels, field_dim)
        self.address_cnn = CharCNN(char_vocab_size, char_dim, channels, field_dim)
        self.legal_embedding = nn.Embedding(legal_vocab_size, legal_dim, padding_idx=0)
        self.mlp = nn.Sequential(
            nn.Linear(field_dim * 4 + legal_dim, 256),
            nn.LayerNorm(256), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(256, output_dim),
        )

    def forward(self, batch: dict[str, torch.Tensor]) -> torch.Tensor:
        legal_ids = batch["legal_form"]
        legal_vectors = self.legal_embedding(legal_ids)
        legal_mask = legal_ids.ne(0).unsqueeze(-1)
        legal = (legal_vectors * legal_mask).sum(1) / legal_mask.sum(1).clamp_min(1)
        features = torch.cat([
            self.name_cnn(batch["name_norm"]),
            self.name_cnn(batch["name_ascii"]),
            self.address_cnn(batch["address_norm"]),
            self.address_cnn(batch["country_norm"]),
            legal,
        ], dim=1)
        return F.normalize(self.mlp(features), p=2, dim=1)


def multi_positive_infonce_loss(query_embeddings, candidate_embeddings,
                                query_ids, candidate_owner_ids, temperature: float = 0.07):
    logits = (query_embeddings @ candidate_embeddings.T) / temperature
    positive = query_ids[:, None].eq(candidate_owner_ids[None, :])
    if not positive.any(dim=1).all():
        raise ValueError("Every query needs at least one positive candidate in its batch.")
    positive_score = torch.logsumexp(logits.masked_fill(~positive, -torch.inf), dim=1)
    all_score = torch.logsumexp(logits, dim=1)
    return (all_score - positive_score).mean()


def cosine_triplet_loss(query_embedding, positive_embedding, negative_embedding, margin: float = 0.2):
    positive_similarity = (query_embedding * positive_embedding).sum(1)
    negative_similarity = (query_embedding * negative_embedding).sum(1)
    return F.relu(margin + negative_similarity - positive_similarity).mean()


def records_to_batch(records: pd.DataFrame, tokenizer: CharTokenizer,
                     legal_vocab: LegalFormVocab, device, name_length: int = 112,
                     address_length: int = 224, country_length: int = 32):
    return {
        "name_norm": tokenizer.encode(records["name_norm"], name_length).to(device),
        "name_ascii": tokenizer.encode(records["name_ascii"], name_length).to(device),
        "address_norm": tokenizer.encode(records["address_norm"], address_length).to(device),
        "country_norm": tokenizer.encode(records["country_norm"], country_length).to(device),
        "legal_form": legal_vocab.encode(records["legal_form"]).to(device),
    }


class PositivePairDataset(Dataset):
    def __init__(self, positives: pd.DataFrame, records: pd.DataFrame, negatives: pd.DataFrame):
        self.left = records.loc[positives.source1_entity_id].reset_index(drop=True)
        self.positive = records.loc[positives.target_entity_id].reset_index(drop=True)
        negative_lookup = defaultdict(list)
        for row in negatives.itertuples(index=False):
            negative_lookup[row.source1_entity_id].append(row.target_entity_id)
        self.negative_ids = [values[0] if values else None for values in
                             (negative_lookup[q] for q in positives.source1_entity_id)]
        self.records = records

    def __len__(self):
        return len(self.left)

    def __getitem__(self, index):
        negative = self.records.loc[self.negative_ids[index]] if self.negative_ids[index] else None
        return self.left.iloc[index], self.positive.iloc[index], negative


def collate_pairs(items):
    left, positive, negative = zip(*items)
    negative_indices = [index for index, row in enumerate(negative) if row is not None]
    return (pd.DataFrame(left), pd.DataFrame(positive),
            pd.DataFrame([row for row in negative if row is not None]), negative_indices)


def split_by_source1(positive_labels, validation_fraction: float = 0.2, seed: int = 42):
    ids = np.array(sorted(positive_labels.source1_entity_id.unique()))
    rng = np.random.default_rng(seed)
    rng.shuffle(ids)
    validation_ids = set(ids[:max(1, int(len(ids) * validation_fraction))])
    return (positive_labels[~positive_labels.source1_entity_id.isin(validation_ids)].copy(),
            positive_labels[positive_labels.source1_entity_id.isin(validation_ids)].copy())


@torch.inference_mode()
def generate_embeddings(model, tokenizer, legal_vocab, records: pd.DataFrame, device, batch_size: int = 512):
    output = []
    model.eval()
    for start in range(0, len(records), batch_size):
        batch = records_to_batch(records.iloc[start:start + batch_size], tokenizer, legal_vocab, device)
        output.append(model(batch).cpu())
    return torch.cat(output).numpy() if output else np.empty((0, 128), dtype=np.float32)

print("[INFO] Model architecture and helper functions loaded.")


[INFO] Model architecture and helper functions loaded.


In [3]:
# ==============================================================================
# Fast Data Loading (Optimized Columns & Memory)
# ==============================================================================
COLS = ["entity_id", "name_norm", "name_ascii", "address_norm", "country_norm", "legal_form"]

print("[INFO] 1. Loading Source 1...")
t0 = time.time()
df_s1 = pd.read_csv(SOURCE1_PATH, sep="\t", usecols=COLS, dtype=str).fillna("")
print(f"       Done in {time.time()-t0:.1f}s | Rows: {len(df_s1):,}")

print("[INFO] 2. Loading Source 2...")
t0 = time.time()
df_s2 = pd.read_csv(SOURCE2_PATH, sep="\t", usecols=COLS, dtype=str).fillna("")
print(f"       Done in {time.time()-t0:.1f}s | Rows: {len(df_s2):,}")

print("[INFO] 3. Loading Source 3...")
t0 = time.time()
df_s3 = pd.read_csv(SOURCE3_PATH, sep="\t", usecols=COLS, dtype=str).fillna("")
print(f"       Done in {time.time()-t0:.1f}s | Rows: {len(df_s3):,}")

print("[INFO] 4. Loading Labeled Pairs...")
t0 = time.time()
df_labels = pd.read_csv(LABELS_PATH, sep="\t", nrows=MAX_LABEL_ROWS, dtype=str, keep_default_na=False)
df_labels["label"] = df_labels["label"].astype(float).astype(int)
print(f"       Done in {time.time()-t0:.1f}s | Rows: {len(df_labels):,}")

print("[INFO] Combining records in memory...")
t0 = time.time()
records = pd.concat([df_s1, df_s2, df_s3], ignore_index=True)
del df_s1, df_s2, df_s3  # Free memory
records = records.drop_duplicates("entity_id").set_index("entity_id", drop=False)

labels = df_labels[
    df_labels["source1_entity_id"].isin(records.index) & 
    df_labels["target_entity_id"].isin(records.index)
].copy()
del df_labels

print(f"[INFO] Dataset ready in {time.time()-t0:.1f}s")
print(f"       Total Unique Records : {len(records):,}")
print(f"       Total Labeled Pairs  : {len(labels):,}")
print(f"       Positive Pairs (1)   : {int((labels.label == 1).sum()):,}")
print(f"       Negative Pairs (0)   : {int((labels.label == 0).sum()):,}")


[INFO] 1. Loading Source 1...
       Done in 5.5s | Rows: 2,206,821
[INFO] 2. Loading Source 2...
       Done in 13.9s | Rows: 5,034,616
[INFO] 3. Loading Source 3...
       Done in 18.9s | Rows: 5,285,603
[INFO] 4. Loading Labeled Pairs...
       Done in 30.9s | Rows: 44,526,560
[INFO] Combining records in memory...
[INFO] Dataset ready in 69.1s
       Total Unique Records : 12,527,040
       Total Labeled Pairs  : 44,526,560
       Positive Pairs (1)   : 7,638,364
       Negative Pairs (0)   : 36,888,196


In [12]:
# # ==============================================================================
# # Model Training (GPU Mixed Precision + Deferred Disk Writes)
# # ==============================================================================
# positives = labels[labels.label == 1].copy()
# negatives = labels[labels.label == 0].copy()

# if positives.empty:
#     raise ValueError("At least one positive pair is required.")

# train_pos, validation_pos = split_by_source1(positives)
# train_qids = set(train_pos.source1_entity_id)
# train_neg = negatives[negatives.source1_entity_id.isin(train_qids)]
# validation_neg = negatives[~negatives.source1_entity_id.isin(train_qids)]

# train_record_ids = list(set(train_pos.source1_entity_id) | set(train_pos.target_entity_id))
# train_records = records.loc[train_record_ids]

# print("[INFO] Fitting Tokenizer and Legal Form Vocabularies...")
# tokenizer = CharTokenizer.fit(train_records)
# legal_vocab = LegalFormVocab.fit(train_records)

# model = EntityEncoder(len(tokenizer.vocab) + 2, len(legal_vocab.vocab) + 2).to(device)
# optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
# scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")

# loaders = {
#     "train": DataLoader(PositivePairDataset(train_pos, records, train_neg), batch_size=BATCH_SIZE,
#                         shuffle=True, collate_fn=collate_pairs, num_workers=2, pin_memory=(device.type == "cuda")),
#     "validation": DataLoader(PositivePairDataset(validation_pos, records, validation_neg), batch_size=BATCH_SIZE,
#                              shuffle=False, collate_fn=collate_pairs, num_workers=2, pin_memory=(device.type == "cuda")),
# }

# print(f"\n[INFO] Starting Training on Device: {device}")
# print(f"[INFO] Train Batches: {len(loaders['train']):,} | Validation Batches: {len(loaders['validation']):,}")
# print(f"[INFO] Total Epochs: {EPOCHS} | Batch Size: {BATCH_SIZE} | Learning Rate: {LEARNING_RATE}\n")

# history = []
# best_loss = float("inf")
# best_state_dict = None
# best_epoch = 1

# for epoch in range(1, EPOCHS + 1):
#     values = {}
#     for stage, loader in loaders.items():
#         training = stage == "train"
#         model.train(training)
#         losses = []
#         desc = f"Epoch [{epoch:02d}/{EPOCHS:02d}] {stage.capitalize()}"
#         pbar = tqdm(loader, desc=desc, leave=False, dynamic_ncols=True)
#         for left, positive, negative, negative_indices in pbar:
#             if training:
#                 optimizer.zero_grad(set_to_none=True)
#             with torch.set_grad_enabled(training), torch.autocast(device_type=device.type, enabled=device.type == "cuda"):
#                 q = model(records_to_batch(left, tokenizer, legal_vocab, device))
#                 p = model(records_to_batch(positive, tokenizer, legal_vocab, device))
#                 owner = torch.as_tensor(left.entity_id.astype("category").cat.codes.to_numpy(copy=True), device=device)
#                 candidate_owner = owner
#                 loss = multi_positive_infonce_loss(q, p, owner, candidate_owner)
#                 if not negative.empty:
#                     n = model(records_to_batch(negative, tokenizer, legal_vocab, device))
#                     selected = torch.as_tensor(negative_indices, device=device)
#                     loss = loss + 0.25 * cosine_triplet_loss(q[selected], p[selected], n)
#             if training:
#                 scaler.scale(loss).backward()
#                 scaler.unscale_(optimizer)
#                 torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
#                 scaler.step(optimizer)
#                 scaler.update()
#             batch_loss = float(loss.detach().cpu())
#             losses.append(batch_loss)
#             pbar.set_postfix({"batch_loss": f"{batch_loss:.4f}", "avg_loss": f"{np.mean(losses):.4f}"})
#         values[f"{stage}_loss"] = float(np.mean(losses))
#     values["epoch"] = epoch
#     history.append(values)
    
#     if values["validation_loss"] < best_loss:
#         best_loss = values["validation_loss"]
#         best_epoch = epoch
#         best_state_dict = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
#         best_marker = "[BEST SO FAR]"
#     else:
#         best_marker = ""
        
#     print(f"Epoch [{epoch:02d}/{EPOCHS:02d}]  |  "
#           f"Train Loss: {values['train_loss']:.4f}  |  "
#           f"Val Loss: {values['validation_loss']:.4f}  {best_marker}")

# # ==============================================================================
# # Save Weights & Artifacts (Only once after complete training)
# # ==============================================================================
# print(f"\n[INFO] Complete training finished. Saving final weights to: {OUTPUT_DIR}")
# if best_state_dict is not None:
#     torch.save({"model": best_state_dict, "epoch": best_epoch, "best_loss": best_loss}, OUTPUT_DIR / "best.pt")
# torch.save({"model": model.state_dict(), "epoch": EPOCHS, "history": history}, OUTPUT_DIR / "last.pt")
# tokenizer.save(OUTPUT_DIR / "tokenizer.json")
# legal_vocab.save(OUTPUT_DIR / "legal_form_vocab.json")
# (OUTPUT_DIR / "config.json").write_text(json.dumps({"output_dim": 128}, indent=2), encoding="utf-8")
# (OUTPUT_DIR / "history.json").write_text(json.dumps(history, indent=2), encoding="utf-8")

# # Create zip archive for 1-click download from Kaggle outputs
# zip_path = Path("/kaggle/working/two_tower_model")
# shutil.make_archive(str(zip_path), 'zip', OUTPUT_DIR)
# print(f"[INFO] Created zip archive for download: {zip_path}.zip")
# print("[INFO] Model weights and vocabularies saved successfully.")


In [5]:
# ==============================================================================
# Load Saved Weights from Disk (Verification)
# ==============================================================================
print(f"[INFO] Loading saved weights from: {OUTPUT_DIR.resolve()}")

# 1. Load Vocabularies
eval_tokenizer = CharTokenizer.load(OUTPUT_DIR / "tokenizer.json")
eval_legal_vocab = LegalFormVocab.load(OUTPUT_DIR / "legal_form_vocab.json")

# 2. Re-initialize Model & Load State Dict
eval_model = EntityEncoder(
    char_vocab_size=len(eval_tokenizer.vocab) + 2,
    legal_vocab_size=len(eval_legal_vocab.vocab) + 2
).to(device)

checkpoint = torch.load(OUTPUT_DIR / "best.pt", map_location=device)
eval_model.load_state_dict(checkpoint["model"])
eval_model.eval()

print(f"[INFO] Successfully loaded model checkpoint from Best Epoch: {checkpoint.get('epoch', 'N/A')}")
print(f"[INFO] Best Validation Loss: {checkpoint.get('best_loss', 'N/A'):.4f}")


[INFO] Loading saved weights from: C:\Users\ps302\OneDrive\Desktop\Amazon\business-entity-resolution\data\model_output
[INFO] Successfully loaded model checkpoint from Best Epoch: 4
[INFO] Best Validation Loss: 0.0022


C:\Users\ps302\AppData\Local\Temp\ipykernel_23436\3380833075.py:16: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(OUTPUT_DIR / "best.pt", map_locatio

In [ ]:
# !pip install faiss-cpu

In [15]:
# import faiss
# import gc

# print("[INFO] Preparing candidate records for FAISS streaming...")

# target_ids = labels["target_entity_id"].unique()
# candidates_df = records.loc[records.index.isin(target_ids)].copy()
# total_candidates = len(candidates_df)
# print(f"[INFO] Total candidate records to index: {total_candidates:,}")

# # Initialize FAISS Index (Exact Inner Product = Cosine Similarity)
# embedding_dim = 128
# faiss_index = faiss.IndexFlatIP(embedding_dim)

# # Save candidate entity IDs to disk to save RAM
# candidate_ids = candidates_df["entity_id"].to_numpy()
# np.save(OUTPUT_DIR / "candidate_ids.npy", candidate_ids)

# # Stream in chunks of 50,000 to keep RAM usage under 1.5 GB
# CHUNK_SIZE = 50000
# INFERENCE_BATCH_SIZE = 1024

# print(f"\n[INFO] Streaming {total_candidates:,} candidates into FAISS in chunks of {CHUNK_SIZE:,}...")
# t_start = time.time()

# eval_model.eval()
# with torch.inference_mode():
#     for start_idx in range(0, total_candidates, CHUNK_SIZE):
#         end_idx = min(start_idx + CHUNK_SIZE, total_candidates)
#         chunk_df = candidates_df.iloc[start_idx:end_idx]
        
#         # Batch inference on GPU
#         chunk_embeddings = []
#         for b_start in range(0, len(chunk_df), INFERENCE_BATCH_SIZE):
#             batch_slice = chunk_df.iloc[b_start:b_start + INFERENCE_BATCH_SIZE]
#             batch_inputs = records_to_batch(batch_slice, eval_tokenizer, eval_legal_vocab, device)
            
#             with torch.autocast(device_type=device.type, enabled=device.type == "cuda"):
#                 batch_emb = eval_model(batch_inputs)
            
#             chunk_embeddings.append(batch_emb.cpu().numpy().astype(np.float32))
        
#         # Add directly to FAISS C++ memory
#         chunk_matrix = np.concatenate(chunk_embeddings, axis=0)
#         faiss_index.add(chunk_matrix)
        
#         # Free memory immediately
#         del chunk_embeddings, chunk_matrix
#         gc.collect()
#         if device.type == "cuda":
#             torch.cuda.empty_cache()
            
#         pct = (end_idx / total_candidates) * 100
#         print(f"  Indexed [{end_idx:,}/{total_candidates:,}] ({pct:.1f}%) | FAISS Total: {faiss_index.ntotal:,}")

# t_elapsed = time.time() - t_start
# print(f"\n[SUCCESS] FAISS indexing complete in {t_elapsed:.1f}s!")

# # Save FAISS Index to disk
# faiss_file = OUTPUT_DIR / "candidate_index.faiss"
# faiss.write_index(faiss_index, str(faiss_file))
# print(f"[INFO] FAISS index saved to disk: {faiss_file}")


In [7]:
import numpy as np
import faiss
from pathlib import Path

# Paths define karein
CANDIDATE_IDS_PATH = Path(r'C:\Users\ps302\OneDrive\Desktop\Amazon\business-entity-resolution\data\model_output\candidate_ids.npy')
FAISS_INDEX_PATH = Path(r'C:\Users\ps302\OneDrive\Desktop\Amazon\business-entity-resolution\data\model_output\candidate_index.faiss')

# 1. Load Candidate IDs
print("Loading Candidate IDs...")
candidate_ids = np.load(CANDIDATE_IDS_PATH, allow_pickle=True)

# 2. Reconstruct candidates_df as a Pandas DataFrame
# Ye line IDs ke basis par global 'records' DataFrame se original DataFrame bana degi
print("Reconstructing candidates_df DataFrame...")
candidates_df = records.loc[candidate_ids].copy()

# 3. Load FAISS Index
print("Loading FAISS Index...")
faiss_index = faiss.read_index(str(FAISS_INDEX_PATH))

print(f"Loaded successfully! Total indexed candidates: {len(candidate_ids):,}")

Loading Candidate IDs...
Reconstructing candidates_df DataFrame...
Loading FAISS Index...
Loaded successfully! Total indexed candidates: 9,895,200


In [9]:
import time
import torch

def predict_matches_faiss(query_id: str, top_k: int = 5):
    """Retrieves Top-K candidate matches and tracks time taken for each sub-task."""
    
    # [TIMER] Overall Total Time Start
    t_total_start = time.perf_counter()
    
    if query_id not in records.index:
        print(f"[ERROR] Query ID '{query_id}' not found in records.")
        return None

    # 1. Data Lookup Time
    t0 = time.perf_counter()
    query_row = records.loc[[query_id]]
    t_lookup = (time.perf_counter() - t0) * 1000

    eval_model.eval()
    
    with torch.inference_mode():
        # 2. Tokenization & Batch Preparation Time
        t0 = time.perf_counter()
        batch_inputs = records_to_batch(query_row, eval_tokenizer, eval_legal_vocab, device)
        t_batching = (time.perf_counter() - t0) * 1000
        
        # 3. Neural Network (Embedding Generation) Time
        t0 = time.perf_counter()
        with torch.autocast(device_type=device.type, enabled=device.type == "cuda"):
            query_emb = eval_model(batch_inputs).cpu().numpy().astype(np.float32)
        t_embedding = (time.perf_counter() - t0) * 1000

    # 4. FAISS Index Search Time
    t0 = time.perf_counter()
    scores, indices = faiss_index.search(query_emb, top_k)
    t_faiss = (time.perf_counter() - t0) * 1000

    scores = scores[0]
    indices = indices[0]

    # 5. Formatting & Ground Truth Lookup Time
    t0 = time.perf_counter()
    actual_positives = set(
        labels[(labels["source1_entity_id"] == query_id) & (labels["label"] == 1)]["target_entity_id"]
    )

    q = query_row.iloc[0]
    print("\n" + "=" * 70)
    print(f"QUERY ID   : {query_id}")
    print(f"Name       : {q.get('name_norm', '')}")
    print(f"Address    : {q.get('address_norm', '')}")
    print(f"Country    : {q.get('country_norm', '')}")
    print("=" * 70)
    
    # ------------------ TIMING SUMMARY ------------------
    t_total = (time.perf_counter() - t_total_start) * 1000
    print("[⏱️ PERFORMANCE TIMING SUMMARY]")
    print(f"  1. Data Lookup Time      : {t_lookup:.2f} ms")
    print(f"  2. Tokenization Time     : {t_batching:.2f} ms")
    print(f"  3. GPU Embedding Time    : {t_embedding:.2f} ms")
    print(f"  4. FAISS Search Time     : {t_faiss:.2f} ms")
    print(f"  Total Execution Time     : {t_total:.2f} ms")
    print("=" * 70)
    print(f"FAISS TOP {top_k} PREDICTED MATCHES:")
    print("=" * 70)

    for rank, (cand_idx, score) in enumerate(zip(indices, scores), start=1):
        cand_id = candidate_ids[cand_idx]
        cand_row = candidates_df.iloc[cand_idx]
        match_flag = "[GROUND TRUTH MATCH]" if cand_id in actual_positives else ""

        print(f"Rank {rank} | Score: {score:.4f} {match_flag}")
        print(f"  ID        : {cand_id}")
        print(f"  Name      : {cand_row.get('name_norm', '')}")
        print(f"  Address   : {cand_row.get('address_norm', '')}")
        print(f"  Country   : {cand_row.get('country_norm', '')}")
        print(f"  Legal Form: {cand_row.get('legal_form', '')}")
        print("-" * 70)


# Test with sample positive query
sample_query = labels[labels["label"] == 1].iloc[6]["source1_entity_id"]
predict_matches_faiss(sample_query, top_k=5)



QUERY ID   : S1-773889195
Name       : prime money
Address    : 17560 ellis road tahlequah ok
Country    : us
[⏱️ PERFORMANCE TIMING SUMMARY]
  1. Data Lookup Time      : 17.76 ms
  2. Tokenization Time     : 17.33 ms
  3. GPU Embedding Time    : 86.40 ms
  4. FAISS Search Time     : 3063.38 ms
  Total Execution Time     : 6114.61 ms
FAISS TOP 5 PREDICTED MATCHES:
Rank 1 | Score: 0.9937 [GROUND TRUTH MATCH]
  ID        : S2-374107005
  Name      : prime money
  Address   : tahlequah ok 0017560 ellis rd
  Country   : us
  Legal Form: missing
----------------------------------------------------------------------
Rank 2 | Score: 0.9923 [GROUND TRUTH MATCH]
  ID        : S3-161452820
  Name      : prime money
  Address   : 17560 ellis rd tahlequah oklahoma
  Country   : us
  Legal Form: incorporated
----------------------------------------------------------------------
Rank 3 | Score: 0.9916 [GROUND TRUTH MATCH]
  ID        : S3-189140669
  Name      : prime money
  Address   : 17560 elli

In [11]:
import time
import torch

def predict_matches_faiss(query_id: str, top_k: int = 10):
    """Retrieves Top-K candidate matches and tracks time taken for each sub-task."""
    
    # [TIMER] Overall Total Time Start
    t_total_start = time.perf_counter()
    
    if query_id not in records.index:
        print(f"[ERROR] Query ID '{query_id}' not found in records.")
        return None

    # 1. Data Lookup Time
    t0 = time.perf_counter()
    query_row = records.loc[[query_id]]
    t_lookup = (time.perf_counter() - t0) * 1000

    eval_model.eval()
    
    # torch.no_grad() is safer on Intel GPUs than inference_mode()
    with torch.no_grad():
        # 2. Tokenization & Batch Preparation Time
        t0 = time.perf_counter()
        batch_inputs = records_to_batch(query_row, eval_tokenizer, eval_legal_vocab, device)
        t_batching = (time.perf_counter() - t0) * 1000
        
        # 3. Neural Network (Embedding Generation) Time
        t0 = time.perf_counter()
        with torch.autocast(device_type=device.type, enabled=device.type == "cuda"):
            query_emb = eval_model(batch_inputs).cpu().numpy().astype(np.float32)
        t_embedding = (time.perf_counter() - t0) * 1000

    # 4. FAISS Index Search (Ultra-fast C++ Inner Product search)
    t0 = time.perf_counter()
    scores, indices = faiss_index.search(query_emb, top_k)
    t_faiss = (time.perf_counter() - t0) * 1000

    scores = scores[0]
    indices = indices[0]

    # Ground truth verification
    actual_positives = set(
        labels[(labels["source1_entity_id"] == query_id) & (labels["label"] == 1)]["target_entity_id"]
    )

    q = query_row.iloc[0]
    print("\n" + "=" * 70)
    print(f"QUERY ID   : {query_id}")
    print(f"Name       : {q.get('name_norm', '')}")
    print(f"Address    : {q.get('address_norm', '')}")
    print(f"Country    : {q.get('country_norm', '')}")
    print("=" * 70)
    
    # ------------------ TIMING SUMMARY ------------------
    t_total = (time.perf_counter() - t_total_start) * 1000
    print("[⏱️ PERFORMANCE TIMING SUMMARY]")
    print(f"  1. Data Lookup Time      : {t_lookup:.2f} ms")
    print(f"  2. Tokenization Time     : {t_batching:.2f} ms")
    print(f"  3. GPU Embedding Time    : {t_embedding:.2f} ms")
    print(f"  4. FAISS Search Time     : {t_faiss:.2f} ms")
    print(f"  Total Execution Time     : {t_total:.2f} ms")
    print("=" * 70)
    print(f"FAISS TOP {top_k} PREDICTED MATCHES:")
    print("=" * 70)

    for rank, (cand_idx, score) in enumerate(zip(indices, scores), start=1):
        cand_id = candidate_ids[cand_idx]
        
        # Changed candidates_df.iloc to records.loc to avoid undefined variable error
        cand_row = records.loc[cand_id] if cand_id in records.index else {}
        match_flag = "[GROUND TRUTH MATCH]" if cand_id in actual_positives else ""

        print(f"Rank {rank} | Score: {score:.4f} {match_flag}")
        print(f"  ID        : {cand_id}")
        print(f"  Name      : {cand_row.get('name_norm', '')}")
        print(f"  Address   : {cand_row.get('address_norm', '')}")
        print(f"  Country   : {cand_row.get('country_norm', '')}")
        print(f"  Legal Form: {cand_row.get('legal_form', '')}")
        print("-" * 70)


# ==============================================================================
# Extract 5 Queries DIRECTLY from Source 1 and get 10 Recommendations
# ==============================================================================
print("Getting 5 queries directly from Source 1 data...")

# Finding the first 5 records whose ID starts with 'S1-'
source1_queries = [idx for idx in records.index if str(idx).startswith("S1-")][:5]

for i, query_id in enumerate(source1_queries, start=1):
    print(f"\n---> Running Test {i}/5 <---")
    # Yahan top_k = 10 pass kar rahe hain
    predict_matches_faiss(query_id, top_k=10)


Getting 5 queries directly from Source 1 data...

---> Running Test 1/5 <---

QUERY ID   : S1-925783039
Name       : orelees barbershop
Address    : 1795 westchester drive high point nc
Country    : us
[⏱️ PERFORMANCE TIMING SUMMARY]
  1. Data Lookup Time      : 59.30 ms
  2. Tokenization Time     : 28.38 ms
  3. GPU Embedding Time    : 175.34 ms
  4. FAISS Search Time     : 3580.53 ms
  Total Execution Time     : 10432.04 ms
FAISS TOP 10 PREDICTED MATCHES:
Rank 1 | Score: 0.9999 [GROUND TRUTH MATCH]
  ID        : S2-517291332
  Name      : orelees bárbershop
  Address   : 1795 westchester drive high point nc
  Country   : us
  Legal Form: missing
----------------------------------------------------------------------
Rank 2 | Score: 0.9974 [GROUND TRUTH MATCH]
  ID        : S2-157377754
  Name      : orelees barbershop
  Address   : 1795 westchester drive nc high point
  Country   : us
  Legal Form: missing
----------------------------------------------------------------------
Rank 3 |

In [10]:
# import time
# import pandas as pd
# import numpy as np
# import torch
# from pathlib import Path
# from tqdm.auto import tqdm

# # Output Path
# output_csv = r"/kaggle/working/ann_top_10.csv"
# top_k = 10
# batch_size = 1024

# # Fallback just in case DEVICE is not defined
# try:
#     DEVICE
# except NameError:
#     DEVICE = torch.device('cpu')

# print("1. Preparing positive labels for fast lookup...")
# # Create a fast O(1) lookup set of positive pairs (source1_id, target_id)
# positive_pairs = set(
#     zip(
#         labels[labels["label"] == 1]["source1_entity_id"],
#         labels[labels["label"] == 1]["target_entity_id"]
#     )
# )

# print("2. Extracting all Source 1 IDs from records...")
# # Fast extraction of Source 1 IDs from the records dataframe
# source1_ids = [idx for idx in records.index if str(idx).startswith("S1-")]
# total_queries = len(source1_ids)
# print(f"Total Source 1 queries to process: {total_queries:,}")

# # Prepare output CSV file with headers
# output_path = Path(output_csv)
# output_path.parent.mkdir(parents=True, exist_ok=True)
# with open(output_path, "w", encoding="utf-8") as f:
#     f.write("query_id,candidate_id,label\n")

# eval_model.eval()
# print(f"\nStarting FAISS batch extraction & saving to {output_csv}...")

# # Process in Batches
# with torch.no_grad():
#     for start_idx in tqdm(range(0, total_queries, batch_size), desc="Extracting ANN Top-10"):
#         end_idx = min(start_idx + batch_size, total_queries)
#         batch_query_ids = source1_ids[start_idx:end_idx]
        
#         # 1. Generate Embeddings for batch
#         batch_df = records.loc[batch_query_ids]
#         batch_inputs = records_to_batch(batch_df, eval_tokenizer, eval_legal_vocab, DEVICE)
#         query_embs = eval_model(batch_inputs).cpu().numpy().astype(np.float32)
        
#         # 2. FAISS Search
#         scores, indices = faiss_index.search(query_embs, top_k)
        
#         # 3. Format results and attach Ground Truth Labels
#         batch_results = []
#         for i, q_id in enumerate(batch_query_ids):
#             for rank in range(top_k):
#                 cand_idx = indices[i, rank]
#                 cand_id = candidate_ids[cand_idx]
                
#                 # Check if this pair is a known positive match in our labels
#                 label = 1 if (q_id, cand_id) in positive_pairs else 0
                    
#                 batch_results.append(f"{q_id},{cand_id},{label}\n")
        
#         # 4. Append to CSV
#         with open(output_path, "a", encoding="utf-8") as f:
#             f.writelines(batch_results)

# print("✅ Batch processing complete! Data saved successfully for LightGBM Reranker.")
